# 00 · Set up and understand the hedging study

**Study question:** How much risk reduction does more frequent hedging buy, and when do transaction costs or incorrect hedge-volatility assumptions change that comparison?

This completed setup notebook values three **short European calls**, calculates their opening exposures, generates joint stock-price histories, checks the simulated marginal behavior and correlations, and records the run inputs. Hedge trading and cash-account P&L begin in notebook 01.

JPM, PNC and GS are labels. All inputs are illustrative and are not calibrated market observations. Stock paths follow correlated GBM under the risk-neutral measure Q, with zero dividends and constant rates/volatilities.

Choose a Python environment with `requirements-harness.txt` and **Run All from a fresh kernel**. Both the project root and `Hedging_Study_Notebooks` work as kernel starting directories. Markdown cells explain the purpose, units and interpretation of each calculation.

## The route through the notebook

| Stage | Purpose | Main functions/tools |
|---|---|---|
| 1. Connect to the project | Locate local modules and import packages | `Path`, `sys.path` |
| 2. Define the study | Keep each security's inputs and each correlation pair named | `option_portfolio`, `scenario` |
| 3. Inspect assumptions | Read units and check name alignment | `pandas.DataFrame`, `.loc` |
| 4. Value calls and inspect exposures | Price options, calculate Greeks and size the initial delta offset | `bs_d1_d2`, `bs_price`, `bs_payoff`, `bs_greeks` |
| 5–7. Generate and inspect paths | Simulate exact correlated Q-GBM and plot samples | `Book`, `simulate_paths`, matplotlib |
| 8–9. Check the simulation | Inspect discounted means, volatility and return correlations | NumPy reductions and log returns |
| 10. Record the run | Capture actual inputs, versions and source hashes | `provenance`, `json.dumps` |

The named dictionaries below are the source of inputs for this notebook. The conversion to ordered arrays happens explicitly at the simulator boundary, using the same asset order on every axis. The shared JSON defaults are not reloaded over your dictionaries.

## 1 · Connect the notebook to the project

Locate the project root, add it to Python's module search path, and import the analysis packages. Imports use saved `.py` files; rerunning the Black–Scholes import cell reloads edits made to that module.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('00_study_setup')


## 2 · Define the study with named inputs

Each security owns its contract and pricing inputs. The scenario separately owns path volatilities and named correlation pairs. Rates and volatilities are annual decimals; time is converted using the specified trading calendar. The setup assumes short calls with $100,000 initial underlying notional per name.

## 3 · Inspect assumptions and correlations

Build the correlation matrix with security labels on both axes. Self-correlations are 1, and each named pair fills both mirrored entries. The checks reject unknown names, missing or repeated pairs, invalid bounds and a matrix that is not positive semidefinite.

In the assumptions table, `call units = notional / spot`, and `time_to_expiration_years = days_to_expiration / days_in_trading_year`. At the baseline, each name has 1,000 call units and a maturity of 0.25 years. These quantities are not exchange contract counts.

## Black–Scholes calculations · three short calls

We hold **short European calls** on JPM, PNC and GS. Each function below uses the named security's spot, strike, annual rate, annual pricing volatility and maturity in years.

| Function | Purpose | Output |
|---|---|---|
| `bs_d1_d2(...)` | Calculate Black–Scholes intermediate quantities | `d1`, `d2` |
| `bs_price(..., kind="call")` | Value one long call unit today | Per-unit call price |
| `bs_payoff(ST, K, kind="call")` | Calculate one call unit's payoff at expiry | `max(ST - K, 0)` |
| `bs_greeks(..., kind="call")` | Calculate one long call unit's sensitivities | Delta, gamma, vega, theta and rho |

These functions describe one **long** call unit. A short unit's value, settlement cash flow and Greeks have the opposite signs. The `d1` and `d2` values themselves stay the same. We calculate per-unit results first, then scale by position quantities in section 4.

### Calculate d1, d2 and today's call prices

Loop through the named portfolio, read each option's inputs explicitly, and convert trading sessions to years. Store full precision and round only the displayed table.

`call_price` is the positive model price of one call unit. The short call's per-unit position value is its negative. A sale now at that price would receive the positive premium and create an equal-valued liability. For an already-existing position, the actual entry premium is a separate historical input.

### Calculate per-unit Greeks

`bs_greeks(...)` returns Greeks for **one long call unit**; multiplying by `-1` gives **one short call unit**.

- Delta is option-price change per $1 stock-price increase.
- Gamma is the change in delta per $1 stock-price increase.
- Vega is per 1.00 change in annual volatility. Multiply by `0.01` for a one percentage-point increase.
- Theta is calendar-time decay per year, holding other inputs fixed.
- Rho is per 1.00 change in the annual interest rate. Multiply by `0.01` for a one percentage-point increase.

The following tables retain these raw units. Section 4 scales them to the actual positions and reports vega/rho per percentage point and theta per trading session.

### Calculate payoffs across example expiry prices

A payoff requires an **expiry stock price** `ST`. Actual terminal prices will come from the later GBM simulation. For now, use an editable illustrative grid from 70 to 130 for each name. These are hypothetical scenarios, not forecasts or simulated paths.

`bs_payoff(...)` returns the amount received by a long call holder. One short call unit pays that amount, giving a negative settlement cash flow. These are expiry payoffs, **not P&L**: they exclude the entry premium, financing, stock hedging and transaction costs.

**Checkpoint:** below or at strike, call payoff is zero. Above strike, the short settlement cash flow becomes increasingly negative.

## 4 · Calculate the opening position exposures

Combine the assumptions, prices and **long-call per-unit Greeks** side by side with `pandas.concat(..., axis=1)`. Rows align by security name; `verify_integrity=True` catches duplicate column labels.

For a short position with positive call-unit magnitude $N$:

| Column | Calculation | Interpretation |
|---|---|---|
| `initial_premium` | $NC$ | Model cash received if sold now; an equal liability is created |
| `hedge_shares` | $N\Delta$ | Positive stock holding offsetting short-call delta |
| `option_dollar_delta` | $-N\Delta S_0$ | Multiply by 0.01 for first-order P&L on a +1% stock move |
| `option_gamma_pnl_for_1pct_move` | $-\frac12N\Gamma(0.01S_0)^2$ | Second-order contribution for a 1% stock move |
| `option_vega_per_vol_point` | $-N\nu\times0.01$ | P&L sensitivity to a one percentage-point pricing-volatility increase |

`initial_premium` is **not an expiry payoff**. For an existing trade, the historical premium must be supplied separately to calculate P&L since entry. These are share-equivalent call units, with no exchange contract multiplier or lot rounding.

The stock holding offsets initial delta only. Negative gamma and vega remain. Greeks for different stocks cannot be treated as interchangeable share hedges.

## 5 · Understand the model before calling the simulator

**Purpose:** know what the simulated paths represent.

For each asset $i$, this study assumes

$$
dS_t^{(i)} = r S_t^{(i)}\,dt + \sigma_i S_t^{(i)}\,dW_t^{(i),\mathbb{Q}},
\qquad dW_t^{(i),\mathbb{Q}}dW_t^{(j),\mathbb{Q}} = \rho_{ij}\,dt.
$$

The exact update over one session is

$$
S_{t+\Delta t}^{(i)} = S_t^{(i)}\exp\!\left[
\left(r-\frac{1}{2}\sigma_i^2\right)\Delta t
+ \sigma_i\sqrt{\Delta t}\,Z_i\right],
\qquad \Delta t=1/252.
$$

The standard-normal shocks $Z_i$ are correlated across names within each session. Increments across sessions are independent in this model. The simulator uses `scenario['path_vols']` for $\sigma_i$.

**Write a short explanation below:**

1. What does using $\mathbb{Q}$ imply about the drift, and why is it not a historical expected-return estimate?
2. Why does the exponential update contain the $-\sigma_i^2/2$ term?
3. What should changing correlation affect? What happens to a standalone Black–Scholes call value if its own marginal inputs stay fixed?

**Checkpoint:** you can distinguish the drift of price from the drift of log price. You can also distinguish an exact GBM price update from the discrete rebalancing of a hedge.

**Model interpretation**

Under Q, zero-dividend stock prices have drift equal to the financing rate $r$. This is a pricing-measure simulation, rather than an estimate of historical expected stock returns. The log-price drift is $r-\sigma^2/2$; the correction makes the exponential's expected growth agree with $e^{r\Delta t}$.

Correlation links the stocks' same-session shocks and changes joint portfolio outcomes. With each stock's own inputs held fixed, changing correlation leaves its standalone Black–Scholes price unchanged. Finite-sample marginal estimates can nevertheless move because the generated joint shocks change.

The price update is exact for constant-parameter GBM. Any hedge introduced in later notebooks still trades only at discrete observation times.

## 6 · Generate joint stock-price histories

The source inputs stay in your named dictionaries. The shared simulator expects a `Book` and numerical arrays, so the following cell converts them using `option_underlyings` as the single asset order. Correlation rows and columns are explicitly selected in that order.

This simulator uses a common expiry, trading calendar and rate. The cell checks that these agree across names before constructing the book. `pricing_vols` remain in the book; the **path volatilities** are supplied separately to `simulate_paths(...)`.

With 5,000 paths, 63 sessions and three names, the output shape is `(5000, 64, 3)`: **path, date including inception, asset**. The assertions verify shape, starting prices, positivity and finite values.

## 7 · Plot sample joint paths

Plot the first 12 simulated histories for each stock, keeping the same simulated path numbers across panels. Transposing the selected array places dates on the first axis, allowing matplotlib to draw one line per path.

Every line begins at its configured spot. A common price scale makes dispersion easier to compare. Twelve example histories provide a visual check; the full simulated sample supplies the statistical diagnostics below.

## 8 · Check the discounted stock mean

**Purpose:** test a consequence of the zero-dividend Q model using Monte Carlo uncertainty.

Under these assumptions,

$$
\mathbb{E}^{\mathbb{Q}}[e^{-rT}S_T^{(i)}]=S_0^{(i)}.
$$

**Calculation steps implemented below**

1. Select terminal prices for all paths and names.
2. Discount them by the common factor $e^{-rT}$ and store the resulting array as `discounted`.
3. Calculate the mean across paths for each asset.
4. Estimate each mean's standard error using sample standard deviation divided by $\sqrt{N}$; use `ddof=1`.
5. Build a table with initial spot, discounted terminal mean, mean standard error and the mean-minus-spot discrepancy. Optionally express that discrepancy in standard-error units.

**Functions/tools:** `np.exp(...)`, `.mean(axis=0)`, `.std(axis=0, ddof=1)`, `np.sqrt(...)`, `pd.DataFrame(...)`.

**Checkpoint:** compare the discrepancy with its Monte Carlo standard error, rather than expecting exact equality to the initial spot. A roughly two-standard-error interval is a useful approximate diagnostic, not a guaranteed pass condition; occasional deviations are expected even under a correct model.

**Explain:** how does standard error change as the number of paths grows? Why is it different from the standard deviation of individual terminal stock outcomes?

The terminal-payoff table also evaluates `bs_payoff` on the simulated final stock prices. Discounted mean payoff estimates should agree with the displayed Black–Scholes prices within sampling uncertainty when path and pricing volatilities match. If those volatilities differ, the two quantities describe different model assumptions.


## 9 · Check correlation using log returns

**Purpose:** check whether the simulated joint shocks match the intended dependence.

**Calculation steps implemented below**

1. Take logs of the price array and difference along the date axis. Store the result as `log_returns`.
2. Confirm the default return-array shape is `(5000, 63, 3)`.
3. Pool path/session observations into a two-dimensional array while preserving the three asset columns.
4. Compute the empirical correlation matrix and display it with asset labels.
5. Compare it with the configured correlation matrix, optionally displaying their difference.

**Functions/tools:** `np.log(...)`, `np.diff(..., axis=1)`, `.reshape(...)`, `np.corrcoef(...)`, `pd.DataFrame(...)`.

**Hint:** `np.corrcoef` treats rows as variables by default. Either transpose your pooled observations or specify `rowvar=False`.

**Checkpoint:** both matrices have ones on the diagonal and similar off-diagonal entries. Finite simulation samples will not match the targets exactly. Pooling sessions is appropriate here because model parameters are constant and Brownian increments are independent across sessions.

**Interpretation prompt:** why are correlations of stock-price levels a different diagnostic from correlations of one-session log returns? Why can changes in correlation affect portfolio outcomes while leaving marginal pricing inputs unchanged?

## 10 · Record exactly what ran

`provenance(...)` records the actual book, ordered simulation inputs, seed, path count, package versions and hashes of the reused source files. The record also includes the original named inputs and the short-call convention.

The pair-keyed correlation dictionary uses Python tuples, which JSON cannot use as object keys. For recording, each pair is represented as a labelled list entry. The computational `scenario` dictionary is left intact.

A hash of the notebook's code cells identifies the implemented experiment without depending on output images or execution counts. This cell displays the record; it does not export a separate file. Save this notebook and keep the record alongside any later exported results.

## Before moving to notebook 01

This notebook has calculated per-unit call values and Greeks, signed short-position exposures, proposed initial delta-offsetting stock holdings, and reproducible joint stock-price histories. The output tables show the Monte Carlo sampling discrepancy in discounted stock means, the simulated return correlations and the marginal volatility estimates.

Equal initial notional does not produce equal premiums or Greeks because the three pricing volatilities differ. Positive stock holdings offset short-call delta at inception; negative option gamma and vega remain. No stock trades, financing ledger or hedged P&L distribution have been simulated here.

The next notebook, **01_single_option_ledger.ipynb**, traces the premium, stock hedge, cash financing, rebalancing and final settlement for one short call. It uses an independently editable named input snapshot matching this setup when prepared; carry forward any later input changes before comparing results between notebooks.

Future research decisions include replacing illustrative inputs with dated estimates, choosing evaluation scenarios before optimizing policies, and adding dividends, exercise, option spreads and margin for a listed-market application.